# Random Numbers and Reproducibility

Random numbers come up all the time in data science — shuffling your data before splitting it, sampling a subset, running simulations. NumPy has a built-in random number generator that handles all of this.

But there's a twist: sometimes you *want* randomness, and sometimes you need your random results to be exactly repeatable. We'll cover both.

In [ ]:
import numpy as np

## Creating a random number generator

NumPy's recommended approach is to create a *generator* object, and then use it to produce random numbers.

In [ ]:
rng = np.random.default_rng()

# one random float between 0 and 1
print("One float:", rng.random())

# five random floats
print("Five floats:", rng.random(5))

Every time you run the cell above, you'll get different numbers. That's expected — we haven't told it to be repeatable yet.

## Random integers

In [ ]:
rng = np.random.default_rng()

# one die roll (1 to 6)
# note: the upper bound is EXCLUDED, so we use 7 to include 6
print("Die roll:", rng.integers(1, 7))

# 10 die rolls
print("10 rolls:", rng.integers(1, 7, size=10))

# a 3x4 grid of random numbers between 0 and 100
grid = rng.integers(0, 101, size=(3, 4))
print("Grid:")
print(grid)

## Random floats in a range

In [ ]:
rng = np.random.default_rng()

# uniform: random floats in a specific range
# say, random temperatures between 20 and 35
temps = rng.uniform(20, 35, size=7)
print("Random temps:", np.round(temps, 1))

## The normal distribution (bell curve)

This is the most important distribution in statistics. Heights, test scores, measurement errors — lots of real-world data roughly follows a bell curve.

It's defined by two numbers:
- **mean** — the center of the curve
- **standard deviation (std)** — how spread out it is

In [ ]:
rng = np.random.default_rng(42)

# generate 10,000 exam scores with mean=70 and std=10
exam_scores = rng.normal(loc=70, scale=10, size=10000)

# check: the actual mean and std should be close to what we asked for
print(f"Mean: {np.mean(exam_scores):.2f}  (asked for 70)")
print(f"Std:  {np.std(exam_scores):.2f}  (asked for 10)")
print(f"Min:  {np.min(exam_scores):.1f}")
print(f"Max:  {np.max(exam_scores):.1f}")

With 10,000 samples, the results are very close to what we requested. More samples = closer match.

In [ ]:
# standard normal: mean=0, std=1 (so common it has its own shortcut)
print("Standard normal:", np.round(rng.standard_normal(10), 3))

## Shuffling and choosing

Two things you'll do constantly when working with datasets.

In [ ]:
rng = np.random.default_rng(42)

deck = np.arange(1, 11)
print("Original:", deck)

# shuffle modifies the array in-place
rng.shuffle(deck)
print("Shuffled:", deck)

If you want a shuffled version *without* changing the original, use `rng.permutation()` instead:

In [ ]:
original = np.arange(1, 11)
shuffled = rng.permutation(original)

print("Original:", original)   # unchanged
print("Shuffled:", shuffled)

### Picking random elements from an array

In [ ]:
rng = np.random.default_rng(42)
colors = np.array(["red", "blue", "green", "yellow", "purple"])

# pick 3 without replacement (no repeats)
print("No repeats:", rng.choice(colors, size=3, replace=False))

# pick 8 with replacement (repeats allowed)
print("With repeats:", rng.choice(colors, size=8, replace=True))

### Weighted random choice

Sometimes you want some options to be more likely. Pass a `p` array with probabilities (they must add up to 1).

In [ ]:
rng = np.random.default_rng(42)

# loaded die: face 6 appears 50% of the time
faces = np.array([1, 2, 3, 4, 5, 6])
weights = [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]

rolls = rng.choice(faces, size=1000, p=weights)

# see how often each face appeared
for face in faces:
    count = np.sum(rolls == face)
    print(f"  Face {face}: {count} times ({count/10:.1f}%)")

## Seeds — making random results repeatable

Here's the problem: you write code that uses random numbers, and every time you run it you get different results. That makes it impossible to debug, and impossible for someone else to reproduce your work.

The fix is a **seed** — a starting number that locks the sequence of random numbers.

In [ ]:
# without a seed: different every time
print("No seed:")
print("  Run 1:", np.random.default_rng().integers(0, 100, size=5))
print("  Run 2:", np.random.default_rng().integers(0, 100, size=5))

In [ ]:
# with a seed: identical every time
print("With seed=42:")
print("  Run 1:", np.random.default_rng(42).integers(0, 100, size=5))
print("  Run 2:", np.random.default_rng(42).integers(0, 100, size=5))

Same seed = same sequence. Every single time. The actual number you pick for the seed doesn't matter — 42, 0, 999, whatever. What matters is using the *same* seed when you want the *same* results.

In practice, create one generator at the top and use it everywhere:

In [ ]:
rng = np.random.default_rng(seed=123)

# all of these will produce the same results every run
data = rng.normal(0, 1, size=100)
indices = rng.integers(0, 100, size=10)
shuffled = rng.permutation(50)

print("Data (first 5):", np.round(data[:5], 3))
print("Indices:", indices[:5])
print("Shuffled:", shuffled[:5])

## The old API (you'll still see this around)

Older tutorials use a different style — functions directly on `np.random`. It works, but the `default_rng()` way is better for new code.

In [ ]:
# old way
np.random.seed(42)
print("Old:", np.random.randint(0, 100, size=5))

# new way (recommended)
rng = np.random.default_rng(42)
print("New:", rng.integers(0, 100, size=5))

The old way uses global state — setting `np.random.seed()` in one function can affect another function's randomness. The new way avoids that by keeping the state inside the `rng` object.

If you see `np.random.rand()`, `np.random.randn()`, or `np.random.randint()` in someone's code, it's the old style. It still works fine, but `default_rng()` is the modern approach.

## Practical examples

Let's use random numbers for two things you'll actually do in practice.

### Simulating coin flips

In [ ]:
rng = np.random.default_rng(42)

# 0 = tails, 1 = heads
flips = rng.integers(0, 2, size=10_000)

heads = np.sum(flips == 1)
tails = np.sum(flips == 0)

print(f"Heads: {heads} ({heads/100:.1f}%)")
print(f"Tails: {tails} ({tails/100:.1f}%)")

### Train/test split

In machine learning, you randomly split your data into a training set and a test set. Here's the basic idea:

In [ ]:
rng = np.random.default_rng(42)

# imagine 100 data samples
n = 100
indices = rng.permutation(n)  # shuffle the indices

# 80% for training, 20% for testing
split = int(0.8 * n)
train_idx = indices[:split]
test_idx = indices[split:]

print(f"Train: {len(train_idx)} samples")
print(f"Test:  {len(test_idx)} samples")
print(f"Train (first 10): {sorted(train_idx[:10])}")
print(f"Test  (first 10): {sorted(test_idx[:10])}")

That covers the random module. Next is the practice notebook — time to use everything we've learned in actual exercises.